# Anomaly detection

Jeet Purohit 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import eli5
from lime.lime_tabular import LimeTabularExplainer
import shap

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import IsolationForest, RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)
from sklearn.base import clone

In [ ]:
df = pd.read_csv("loan_data.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("Data types:")
print(df.dtypes)

In [ ]:
# Data distribution analysis
# GÖR DENNA BÄTTRE ELLER GRAFISKT
# KANSKE FLYTTA TILL SLUTET AV PREPROCCESSING STEGET
features = [
    'person_age', 'person_gender', 'person_education', 'person_income',
    'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent',
    'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
    'credit_score', 'previous_loan_defaults_on_file', 'loan_status'
]

categorical = [
    'person_gender', 'person_education', 'person_home_ownership',
    'loan_intent', 'previous_loan_defaults_on_file'
]
numerical = [f for f in features if f not in categorical]

for col in features:
    print(f"\n {col}")
    if col in numerical:
        print(f"  Min: {df[col].min()} | Max: {df[col].max()} | Mean: {df[col].mean():.2f} | Std: {df[col].std():.2f}")
    else:
        counts = df[col].value_counts()
        total = counts.sum()
        for val, cnt in counts.items():
            pct = cnt / total * 100
            print(f"  {val}: {cnt} ({pct:.1f}%)")

# Data Preproccessing

* Remove unnessesary features: Person_Gender
* The the means for each gender are not far apart, but we think that it is a possibility that the model will learn some bias from the data we have.

In [ ]:
mean_income_by_gender = df.groupby("person_gender")["person_income"].mean()
for gender, mean_income in mean_income_by_gender.items():
    print(f"{gender}: Mean income = {mean_income:.2f}")

* We want to change some of the categorical columns to Numerical values
* We also remove unnessesary features like "person_gender" beacuase this might not be needed

In [ ]:
df = df.drop(columns=["person_gender"])

In [ ]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
print(f"Categorical columns: {cat_cols}\n")

In [ ]:
label_encoders = {}
df_numerical = df.copy()

for col in cat_cols:
    le = LabelEncoder()
    df_numerical[col] = le.fit_transform(df_numerical[col])
    label_encoders[col] = le
    # print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print(f"\nEncoded shape: {df_numerical.shape}")
df_numerical.head()

In [ ]:
for col in cat_cols:
    le = LabelEncoder()
    le.fit(df[col])
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"{col} mapping:")
    for k, v in mapping.items():
        print(f"  {k}: {v}")
    print()

    Feature dependencies and correlations

In [ ]:
# Heatmap of feature correlations
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load dataset (change path or DataFrame if needed)
df = pd.read_csv('loan_data.csv')

# Compute correlation matrix
corr = df_numerical.corr()

# Plot heatmap
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()

# Missing values/incomplete samples

In [ ]:
nulls = df_numerical.isnull().sum()
total_null = nulls.sum()
print("number of null", total_null)

# Outliers
* we se that there are only 7 people over 100 taking loans, we remove this beacause------------------- below cells we first caount all the outliers and then remove them
* 24 åpeople over income person
* 12 people over 50 year sof eperiance, you usially take pension at 65 years old
* we converted float to int in cb_person_cred_hist_length because we saw that there were no float values. 

In [ ]:
# Count samples where person_age > 100

person_age_100 = (df_numerical['person_age'] >= 100).sum()
print(f"Number of samples with person_age >= 100: {person_age_100}")

In [ ]:
person_income_1M = (df_numerical['person_income'] > 1000000).sum()
print(f"Number of samples with person_income > 1M: {person_income_1M}")

In [ ]:
person_emp_exp_over_ = (df_numerical['person_emp_exp'] > 50).sum()
print(f"Number of samples with person_emp_exp > 50: {person_emp_exp_over_}")

In [ ]:
# Check for float columns and show unique values for cb_person_cred_hist_length
# float_cols = df.select_dtypes(include='float')
# print("Float columns:", float_cols.columns.tolist())

# if 'cb_person_cred_hist_length' in df.columns:
#     print("Unique values in cb_person_cred_hist_length:")
print(sorted(df_numerical['cb_person_cred_hist_length'].unique()))

In [ ]:
# Convert all values in cb_person_cred_hist_length to integers
print(df_numerical['cb_person_cred_hist_length'].head())
df_numerical['cb_person_cred_hist_length'] = df_numerical['cb_person_cred_hist_length'].astype(int)

print(df_numerical['cb_person_cred_hist_length'].head())

In [ ]:
# print(f"Before cleaning: {df.shape[0]} rows")

# remove unrealistic ages (e.g. 100+)
df_numerical = df_numerical[df_numerical["person_age"] <= 100]

# remove extreme employment experience outliers
df_numerical = df_numerical[df_numerical["person_emp_exp"] <= 60]

print(f"After cleaning: {df.shape[0]} rows")
print(f"Removed {45000 - df.shape[0]} outlier rows")

# Data preprocessing 
* splitting the data test aand shit

In [ ]:
X = df_numerical.drop(columns=["loan_status"])
y = df_numerical["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# evaulation and model tuning
- tuning 
    * use grid searchCV
    * first run we had alot of diffrent combination in param_grid, this took 4 minutes to run. second run we narrowed the parameter values around the "best fit" parameters. eg if we had max depth 20 in max depth first run. we then change the list so it checks the scond run 18, 20, 23 24, none.

    snotees, change paramets



In [ ]:
print(len(X_train))

# TIME TO RUN 3 mins.
    depends on PC stats 

In [ ]:

# dt_model = DecisionTreeClassifier(random_state=42)
# skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
# scores = ['f1', 'precision', 'recall']

# # Model tuning, finding best parameters --------------------------------------------
# param_grid = {
#     'max_depth': [16, 18, 25, None],
#     'min_samples_split': [7, 10, 12, 20,],
#     'min_samples_leaf': [29, 32, 35, 38],
#     'class_weight': ["balanced", None],
#     'min_weight_fraction_leaf' : [0.0, 0.05],
#     'max_features' : ['sqrt', 'log2', None]
# }

# # !Note! change n_jobs in parameters below depending on how many cores you want to use.
# grid_search = GridSearchCV(estimator=dt_model, param_grid=param_grid, 
#                            cv=10, n_jobs=-1, verbose=2, scoring=scores, refit='f1') 


# grid_search.fit(X_train, y_train)
# tuned_dt = grid_search.best_estimator_ #here we set the best parameters on our model

# best_params = grid_search.best_params_ 
# print(f"Best parameters: ")
# for param, value in best_params.items():
#     print(f"{param}: {value}")

# #-----------------------------------------------------------------------------------------

# # Evaluation across 10 folds with the fine tuned model -----------------------------------
# cv_results = cross_validate(tuned_dt, X_train, y_train, cv=skf, scoring=scores,
#                             return_train_score=False)

# print("scores evaluation phase:\n")
# for metric in scores:
#     scores = cv_results[f"test_{metric}"]
#     print(f"{metric:>10s}:  mean={scores.mean():.4f}  std={scores.std():.4f}")

# #-----------------------------------------------------------------------------------------
# y_pred = tuned_dt.predict(X_test)

# real_f1   = f1_score(y_test, y_pred)
# real_prec = precision_score(y_test, y_pred)
# real_rec  = recall_score(y_test, y_pred)


# print(" scores y_eval vs y_test")

# print(f"F1:       {real_f1:.4f}")
# print(f"Precision:{real_prec:.4f}")
# print(f"Recall:   {real_rec:.4f}")


# copy of above cell
    the below cell is a copy of above, we have added the best parameters from above so that it is quicker to run. but above cell is the complete process with searchSV that we have used

In [ ]:

dt_model = DecisionTreeClassifier(random_state=42)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = ['f1', 'precision', 'recall']

# Model tuning, finding best parameters --------------------------------------------
param_grid = {
    'max_depth': [18],
    'min_samples_split': [7],
    'min_samples_leaf': [35],
    'class_weight': [None],
    'min_weight_fraction_leaf' : [0.0],
    'max_features' : [None]
}

# !Note! change n_jobs in parameters below depending on how many cores you want to use.
grid_search = GridSearchCV(estimator=dt_model, param_grid=param_grid, 
                           cv=10, n_jobs=-1, verbose=2, scoring=scores, refit='f1') 


grid_search.fit(X_train, y_train)
tuned_dt = grid_search.best_estimator_ #here we set the best parameters on our model

best_params = grid_search.best_params_ 
print(f"Best parameters: ")
for param, value in best_params.items():
    print(f"{param}: {value}")

#-----------------------------------------------------------------------------------------

# Evaluation across 10 folds with the fine tuned model -----------------------------------
cv_results = cross_validate(tuned_dt, X_train, y_train, cv=skf, scoring=scores,
                            return_train_score=False)

print("scores evaluation phase:\n")
for metric in scores:
    scores = cv_results[f"test_{metric}"]
    print(f"{metric:>10s}:  mean={scores.mean():.4f}  std={scores.std():.4f}")

#-----------------------------------------------------------------------------------------
y_pred = tuned_dt.predict(X_test)

real_f1   = f1_score(y_test, y_pred)
real_prec = precision_score(y_test, y_pred)
real_rec  = recall_score(y_test, y_pred)


print(" scores y_eval vs y_test")

print(f"F1:       {real_f1:.4f}")
print(f"Precision:{real_prec:.4f}")
print(f"Recall:   {real_rec:.4f}")


# test on unseen data
- this thest is made so we can se if our model perofms similar to above, not similar = overfitting or problem in data splitting

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Rejected", "Approved"]))


# confusion metric
snacka om specificity

In [ ]:
# specificity = TN / (TN + FP)
cm_baseline = confusion_matrix(y_test, y_pred)
test_spec = cm_baseline[0, 0] / (cm_baseline[0, 0] + cm_baseline[0, 1])

print(f"  Specificity: {test_spec:.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Rejected", "Approved"], cmap="Blues", ax=ax)
ax.set_title("Confusion Matrix - Baseline")
plt.tight_layout()
plt.show()

# print tree


In [ ]:
print(f"Total rows: {df_numerical.shape[0]}")

# återkom om nedan och see om ni kan läsa av

In [ ]:
plt.figure(figsize=(250, 90))
plot_tree(tuned_dt, feature_names=X_train.columns, class_names=['Not Approved', 'Approved'], filled=True, max_depth=10)
plt.show()

# feature importance

In [ ]:
importances = pd.Series(tuned_dt.feature_importances_, index=X.columns).sort_values(ascending=False)

print(f"Tree depth: {tuned_dt.get_depth()}")
print(f"Number of leaves: {tuned_dt.get_n_leaves()}\n")
print(importances.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
importances.plot.barh(ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Feature Importance - Decision Tree")
ax.set_xlabel("Importance")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# tree explainer


In [ ]:
explainer = shap.TreeExplainer(tuned_dt)
shap_values = explainer.shap_values(X_test)
shap_values

# data poisoning
- lable flipping
- 

In [ ]:
def label_flipping_attack(X_tr, y_tr, flip_pct, seed=42):
    """Flip a percentage of training labels at random."""
    rng = np.random.RandomState(seed)
    y_poisoned = y_tr.copy()
    n_flip = int(len(y_tr) * flip_pct)
    flip_idx = rng.choice(y_tr.index, size=n_flip, replace=False)
    y_poisoned.loc[flip_idx] = 1 - y_poisoned.loc[flip_idx]
    return y_poisoned, flip_idx

In [ ]:

# flip_rates = [0.05, 0.10, 0.20, 0.70]
flip_rates = [0.05, 0.10, 0.15, 0.20, 0.25]

flip_results = {}

for rate in flip_rates:
    y_poisoned, flipped = label_flipping_attack(X_train, y_train, rate)
    orig_counts = y_train.loc[flipped].value_counts().to_dict()

    dt_poison = clone(tuned_dt) # tuned_dt is the model with the best parametes from gridCV
    dt_poison.fit(X_train, y_poisoned)
    y_pred_p = dt_poison.predict(X_test)

    poisoned_f1   = f1_score(y_test, y_pred_p)
    poisoned_prec = precision_score(y_test, y_pred_p)
    poisoned_rec  = recall_score(y_test, y_pred_p)


    print(flipped) # sample index where we flipped label

    flip_results[rate] = {
        "n_flipped": len(flipped),
        "f1": round(poisoned_f1, 4),
        "precision": round(poisoned_prec, 4),
        "recall": round(poisoned_rec, 4),
        "cm": confusion_matrix(y_test, y_pred_p),
    }

    print(f"Flip rate: {rate*100:.0f}%  ({len(flipped)} labels flipped)")
    print(f"  F1       : {poisoned_f1:.4f}  (baseline {real_f1:.4f}, delta={poisoned_f1 - real_f1:+.4f})")
    print(f"  Precision: {poisoned_prec:.4f} (baseline {real_prec:.4f}, delta={poisoned_prec - real_prec:+.4f})") 
    print(f"  Recall   : {poisoned_rec:.4f} (baseline {real_rec:.4f}, delta={poisoned_prec - real_rec:+.4f})\n")

In [ ]:
fig, axes = plt.subplots(1, len(flip_rates) + 1, figsize=(5 * (len(flip_rates) + 1), 4))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
    display_labels=["Rejected", "Approved"], cmap="Blues", ax=axes[0])

for i, rate in enumerate(flip_rates):
    cm = flip_results[rate]["cm"]
    f1_drop = ((real_f1 - flip_results[rate]['f1']) / real_f1) * 100
    prec_drop = ((real_prec - flip_results[rate]['precision']) / real_prec) * 100
    ConfusionMatrixDisplay(cm, display_labels=["Rejected", "Approved"]).plot(
        cmap="Reds", ax=axes[i + 1])
    axes[i + 1].set_title(
    f"Flip {rate*100:.0f}%\n"
    f"F1={flip_results[rate]['f1']:.3f}  Prec={flip_results[rate]['precision']:.3f}  Rec={flip_results[rate]['recall']:.3f}"
            f"F1={flip_results[rate]['f1']:.3f} (↓{f1_drop:.1f}%)\n"
        f"Prec={flip_results[rate]['precision']:.3f} (↓{prec_drop:.1f}%)")

plt.suptitle("Label Flipping - Confusion Matrices", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
rates_with_baseline = [0] + flip_rates
f1_scores = [real_f1] + [flip_results[r]['f1'] for r in flip_rates]
prec_scores = [real_prec] + [flip_results[r]['precision'] for r in flip_rates]
rec_scores = [real_rec] + [flip_results[r]['recall'] for r in flip_rates]

ax.plot(rates_with_baseline, f1_scores, 'o-', label='F1 Score', linewidth=2)
ax.plot(rates_with_baseline, prec_scores, 's-', label='Precision', linewidth=2)
ax.plot(rates_with_baseline, rec_scores, '^-', label='Recall', linewidth=2)
ax.set_xlabel('Label Flip Rate')
ax.set_ylabel('Score')
ax.set_title('Model Performance vs Label Flip Rate')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# isolation forest

    Normalizing.

In [ ]:
# Normalize X_train using StandardScaler
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train_norm = scaler.fit_transform(X_train)

# Convert back to DataFrame for easier use
x_train_norm = pd.DataFrame(x_train_norm, columns=X_train.columns, index=X_train.index)

x_train_norm.head()

In [ ]:
x_train_norm.std(ddof=0)

### tuning


* search grid want scores, iso forest have no scores due to unsuperviced model. but we cna do it cutsom, we calculate the perfomance by checking how many namoalies OUT OF the true flipped anomalies that the isoforest gets, and witht that we have created our own search. it exhausivly searches all combination and updates the "best score " variable.

Notice: the less max_sampes parameter, the more anomalies WITHIN our poisond flipped it finds???
    
    takes 2 min

In [ ]:
from itertools import product

param_grid2 = {
    'n_estimators': [200, 250],
    'max_samples': [30, 40],
    'contamination': [0.19],
    'bootstrap' : [True,False],
    'max_features' : [4]
}

# Get all combinations of parameters
keys, values = zip(*param_grid2.items())
param_combinations = [dict(zip(keys, v)) for v in product(*values)]

best_score = -1
best_params = None
best_anomaly_indices = None
best_detected_poisoned = None

for params in param_combinations:
    # Create and fit the model
    iso_model = IsolationForest(**params, random_state=42)
    preds = iso_model.fit_predict(x_train_norm)
    anomaly_indices = set(x_train_norm.index[preds == -1])

    # Calculate overlap with flipped indices
    detected_poisoned = anomaly_indices & set(flipped)
    score = len(detected_poisoned)
    # print(f"Params: {params} | Detected poisoned: {score}")

    # Track the best
    if score > best_score:
        best_score = score
        best_params = params
        best_anomaly_indices = anomaly_indices

# below is the values for THE BEST interation that was run above
print("total anomalies detected",len(best_anomaly_indices))
print("\nBest params:", best_params)
print("Max detected poisoned samples:", best_score)


# total anomalies detected 6839

# Best params: {'n_estimators': 200, 'max_samples': 30, 'contamination': 0.19, 'bootstrap': False, 'max_features': 4}
# Max detected poisoned samples: 1750




# fit model with the best params

In [ ]:
print(best_params)
iso_model_tuned = IsolationForest(**best_params, random_state=42)
iso_model_tuned.fit(X_train)

# understanding the decisions
    theu anomalies
    

# trying own build samples


In [ ]:
# Create a sample with mean values of each feature

# ändra samples beroende på vad SHAP säger om varför ISO forest sääger
# nej till lone eller inte, ändra dom värden och se om det påvärkar mycket
# Desisiton tree säger att previous_loan_defaults_on_file påvärkar 
# jättemycket, men ISO forest bryr sig inte om den
sample = {
    "person_age": 27.752,
    "person_education": 1.742415,
    "person_income": 80155.354051,
    "person_emp_exp": 5.402534,
    "person_home_ownership": 1.694043,
    "loan_amnt": 9569.354023,
    "loan_intent": 2.531285,
    "loan_int_rate": 11.005086,
    "loan_percent_income": 0.139420,
    "cb_person_cred_hist_length": 5.871888,
    "credit_score": 632.592937,
    "previous_loan_defaults_on_file": 0 #0.508085
}

sample = pd.DataFrame([sample], columns=X_train.columns)
# Predict with the trained Isolation Forest
result = iso_model_tuned.predict(sample)

print("Prediction (1 = normal, -1 = anomaly):", result[0])

In [ ]:
# results = []

# param_grid2 = {
#     'n_estimators': [250, 300, 350],
#     'max_samples': ['auto', 50, 100, 200, 300],
#     'contamination': ['auto', 0.05, 0.1, 0.15, 0.25],
#     'bootstrap' : [True,False],
# }

# for params in param_combinations:
#     iso_model = IsolationForest(**params, random_state=42)
#     preds = iso_model.fit_predict(X_train)
#     anomaly_indices = set(X_train.index[preds == -1])
#     detected_poisoned = anomaly_indices & set(flipped)
#     score = len(detected_poisoned)
#     total_anomalies = len(anomaly_indices)
#     results.append({
#         'params': params,
#         'score': score,
#         'total_anomalies': total_anomalies,
#         'anomaly_indices': anomaly_indices,
#         'detected_poisoned': detected_poisoned
#     })

# # Find the highest score
# max_score = max(r['score'] for r in results)
# best_results = [r for r in results if r['score'] == max_score]
# # Among those, pick the one with the lowest total anomalies
# best_result = min(best_results, key=lambda r: r['total_anomalies'])

# print("Best params (highest score, lowest anomalies):", best_result['params'])
# print("Max detected poisoned samples:", best_result['score'])
# print("Total anomalies detected:", best_result['total_anomalies'])

    radera anomalies som den hitade som mvi har flippat. blir modellen som vi poisonade bättre?

# nu måste vi kolla så att inte FALs positives har ökat

In [ ]:
# SHAP explanation for Isolation Forest anomalies and normal samples
import shap

# Use KernelExplainer for Isolation Forest (since it's not tree-based)
explainer = shap.KernelExplainer(iso_model_tuned.decision_function, shap.sample(X_train, 100, random_state=42))

# Find indices of one anomaly and one normal sample
anomaly_idx = np.where(iso_preds_full == -1)[0][0]
normal_idx = np.where(iso_preds_full == 1)[0][0]

sample_anomaly = X_train.iloc[[anomaly_idx]]
sample_normal = X_train.iloc[[normal_idx]]

# Compute SHAP values for both
shap_values_anomaly = explainer.shap_values(sample_anomaly, nsamples=100)
shap_values_normal = explainer.shap_values(sample_normal, nsamples=100)

print("Anomaly sample index:", anomaly_idx)
print("Normal sample index:", normal_idx)

# Plot SHAP values for the anomaly
print("\nSHAP explanation for anomaly sample:")
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values_anomaly, sample_anomaly, matplotlib=True)

# Plot SHAP values for the normal sample
print("\nSHAP explanation for normal sample:")
shap.force_plot(explainer.expected_value, shap_values_normal, sample_normal, matplotlib=True)

In [ ]:
import shap

# Wrap anomaly SHAP values
anomaly_expl = shap.Explanation(
    values=shap_values_anomaly[0],
    base_values=explainer.expected_value,
    data=sample_anomaly.values[0],
    feature_names=sample_anomaly.columns.tolist()
)

# Wrap normal SHAP values
normal_expl = shap.Explanation(
    values=shap_values_normal[0],
    base_values=explainer.expected_value,
    data=sample_normal.values[0],
    feature_names=sample_normal.columns.tolist()
)

# Waterfall plots
print("SHAP Waterfall Plot for Anomaly Sample:")
shap.plots.waterfall(anomaly_expl)

print("SHAP Waterfall Plot for Normal Sample:")
shap.plots.waterfall(normal_expl)

iso_model_tuned.decision_function

In [ ]:
# Heatmap of feature correlations
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load dataset (change path or DataFrame if needed)
df = pd.read_csv('loan_data.csv')

# Compute correlation matrix
corr = df_numerical.corr()

# Plot heatmap
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()